# Step 51 Run LOSO Shortlist Stability Checks

## What This Notebook Does

This is the second public Stage-5 notebook. It preserves the shortlist-comparison workflow that:
- reuses the canonical Stage-4 retained dataset
- trains LOSO models for the manuscript-facing shortlist
- refits the selected candidate per fold when needed
- computes cross-fold state matching and stability summaries
- writes the per-K outputs used later for Figure 2 and Table S8 support

## When To Run It

Run this notebook after the broad screening stage in:
- `step50_run_loso_k_sweep_model_selection.ipynb`

This public notebook focuses on the manuscript-facing shortlist comparison:
- `K = 3`
- `K = 5`

## Manuscript Linkage

- Main Methods 2.5.1
- Supplementary Methods 1.6-1.7
- Figure 2 support
- Table S8 support

## Inputs Expected

- the canonical Stage-4 retained dataset root for `intermediate + nolags + minlen15`
- the same Stage-4 `segments_manifest.tsv` used by the Stage-5 selection workflow, unless you provide an explicit override

## Outputs Written

- per-K shortlist folders such as `K03/` and `K05/`
- per-fold saved model artifacts and summaries
- `state_matching_scores.tsv`
- `fold_summaries_table_matched.tsv`
- similarity matrices and transition-summary files
- `invalid_folds.tsv` when a fold has to be excluded from the stability summary

## Environment Notes

This notebook runs the actual TensorFlow and `osl_dynamics` shortlist workflow.
- it is GPU-sensitive in normal use
- CPU fallback is possible, but usually much slower and less practical for the full shortlist run
- chunk/resume settings are exposed because held-out folds may need to be completed across multiple runs

## Public Workflow Note

To keep this notebook readable, the dense shortlist training and matching machinery now lives in descriptive same-directory Python backend code that the public helper calls directly.

## Interpretation Note

This notebook is the manuscript-facing shortlist comparison step, not the full historical shortlist universe.
It keeps the public comparison centered on `K=3` versus `K=5`, while the final manuscript choice remains `K=3`.


In [ ]:
# Step 0. Import Python modules and locate the active Stage-5 helper module

import sys
from pathlib import Path

STAGE5_DIR = Path.cwd()
candidate = Path.cwd() / "notebooks" / "5_hmm_selection"
if not (STAGE5_DIR / "stage5_hmm_selection_helpers.py").exists() and candidate.exists():
    STAGE5_DIR = candidate

if not (STAGE5_DIR / "stage5_hmm_selection_helpers.py").exists():
    raise FileNotFoundError(
        f"Stage-5 helper not found: {STAGE5_DIR / 'stage5_hmm_selection_helpers.py'}"
    )

if str(STAGE5_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE5_DIR.resolve()))

from stage5_hmm_selection_helpers import run_public_shortlist_step


In [ ]:
# Step 1. User-editable roots and shortlist settings
#
# SEGMENTS_ROOT should point to the Stage-4 retained-data root that contains
# the canonical `intermediate/` branch.
# SHORTLIST_OUTPUT_ROOT is where this notebook will write the per-K shortlist
# folders and the cross-fold stability outputs.
# Leave MANIFEST_TSV as None unless you need to override the standard Stage-4
# manifest location on your machine.

SEGMENTS_ROOT = Path("<SET_SEGMENTS_ROOT>")
SHORTLIST_OUTPUT_ROOT = Path("<SET_SHORTLIST_OUTPUT_ROOT>")

DATA_VARIANT = "intermediate"
FEATURE_MODE = "nolags"
MINLEN = 15

# Public manuscript-facing shortlist comparison.
K_LIST = [3, 5]

# Optional explicit manifest override.
MANIFEST_TSV = None

# Chunk/resume and debug controls for long runs.
MAX_NEW_FOLDS_PER_RUN = 1
FORCE_RERUN_HELDOUTS = []
DEBUG_SUBJECTS = None
DEBUG_SEEDS = None

# Keep None to use TensorFlow memory-growth mode.
GPU_MEMORY_LIMIT_MB = None

FINAL_ROOT = SEGMENTS_ROOT / DATA_VARIANT
SHORTLIST_RUN_DIR = SHORTLIST_OUTPUT_ROOT / f"PipelineD_C2_{DATA_VARIANT}_{FEATURE_MODE}_minlen{MINLEN}"


In [ ]:
# Step 2. Validate the configured inputs and print a short run summary

def ensure_configured_path(path_value, label, must_exist=False):
    path = Path(path_value)
    if "<SET_" in str(path):
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")
    if must_exist and not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")
    return path

SEGMENTS_ROOT = ensure_configured_path(SEGMENTS_ROOT, "SEGMENTS_ROOT", must_exist=True)
SHORTLIST_OUTPUT_ROOT = ensure_configured_path(SHORTLIST_OUTPUT_ROOT, "SHORTLIST_OUTPUT_ROOT")

print("Stage 5 / Step 51: shortlist stability and state matching")
print(f"  Canonical Stage-4 root: {FINAL_ROOT}")
print(f"  Output root:            {SHORTLIST_RUN_DIR}")
print(f"  Feature mode:           {FEATURE_MODE}")
print(f"  Minimum segment len:    {MINLEN} TR")
print(f"  Public shortlist:       {K_LIST}")
print(f"  GPU memory cap (MB):    {GPU_MEMORY_LIMIT_MB}")
print("  Packages needed:        tensorflow, osl_dynamics, numpy, pandas, matplotlib, scipy")
print("  Interpretation note:    this is the manuscript-facing shortlist comparison, not the full historical candidate set.")


## Step 3. Run the shortlist stability workflow

The helper called below keeps the original shortlist behavior intact while making this public notebook easier to follow.

The key public idea is simple:
- Step 50 screens the full `K = 2..12` range
- Step 51 compares the manuscript-facing shortlist `K=3` versus `K=5`
- Step 52 rebuilds the compact manuscript summary outputs


In [ ]:
# Step 3a. Run the active Python shortlist backend through the public helper

run_result = run_public_shortlist_step(
    segments_root=SEGMENTS_ROOT,
    shortlist_output_root=SHORTLIST_OUTPUT_ROOT,
    k_list=K_LIST,
    data_variant=DATA_VARIANT,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    manifest_tsv=MANIFEST_TSV,
    max_new_folds_per_run=MAX_NEW_FOLDS_PER_RUN,
    gpu_memory_limit_mb=GPU_MEMORY_LIMIT_MB,
    force_rerun_heldouts=FORCE_RERUN_HELDOUTS,
    debug_subjects=DEBUG_SUBJECTS,
    debug_seeds=DEBUG_SEEDS,
)

print("Using backend module:", run_result["backend_module"])
print("Backend status:", run_result["backend_status"])
if run_result["backend_message"]:
    print(run_result["backend_message"])
print("Resolved Stage-4 root:", run_result["final_root"])
print("Resolved manifest:", run_result["manifest_tsv"])
print("Shortlist output root:", run_result["out_root"])


In [ ]:
# Step 4. Review the key shortlist outputs that feed the manuscript summary

for k, info in run_result["per_k_outputs"].items():
    print(f"\nK={k} output folder: {info['k_dir']}")

    scores_df = info["state_matching_scores_df"]
    if scores_df is not None:
        print(f"K={k} state-matching scores:")
        display(scores_df.head())
    else:
        print(f"K={k} state-matching scores are not available yet.")

    fold_df = info["fold_summaries_table_matched_df"]
    if fold_df is not None:
        print(f"K={k} matched fold summary:")
        display(fold_df.head())
    else:
        print(f"K={k} matched fold summary is not available yet.")

    invalid_df = info["invalid_folds_df"]
    if invalid_df is not None and len(invalid_df) > 0:
        print(f"K={k} invalid-fold audit:")
        display(invalid_df)
